## Creating Catalog and Schemas for Bronze, Silver and gold layers.

In [ ]:
%sql
USE CATALOG ishi_catalog;


CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [ ]:
%sql
use bronze;

In [ ]:
from pyspark.sql.functions import col
import re

# Helper function to clean column names
def clean_column_names(df):
    for c in df.columns:
        new_c = re.sub(r'[^a-zA-Z0-9_]', '_', c)  # Replace invalid chars with _
        df = df.withColumnRenamed(c, new_c)
    return df

# Define paths
pos_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/pos"
schema_loc = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_schemas/pos"
chkpt      = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_checkpoints/pos"
bronze_pos = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/pos"   # <- this was missing earlier

# Auto Loader stream
pos_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(pos_source)
)

# Clean column names
pos_df = clean_column_names(pos_df)

# Write to Bronze Delta
(pos_df.writeStream
    .format("delta")
    .option("checkpointLocation", chkpt)
    .option("path", bronze_pos)
    .trigger(once=True)   # treat like batch
    .outputMode("append")
    .start())


###  Displaying the ingested pos file

In [ ]:
pos_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/pos"

# Read raw POS data (assuming CSV for now, you can change format if needed)
pos_raw_df = (spark.read
              .format("csv")
              .option("header", "true")   # since POS data usually has headers
              .option("inferSchema", "true")
              .load(pos_source)).limit(50)

# Display raw POS data
display(pos_raw_df)


store_id,sku_id,date,units_sold,price,promo_flag
S0014,SKU00146,2025-06-30,5.0,1.13,0.0
S0050,SKU00052,2025-06-30,3.0,29.55,0.0
S0036,SKU00035,2025-06-30,150.37390943082676,1.43,1.0
S0015,SKU00162,2025-06-30,4.0,10.47,0.0
S0042,SKU00166,2025-06-30,3.0,null,1.0
S0042,SKU00071,2025-06-30,6.0,2517.1220398836726,1.0
S0037,SKU00059,INVALID_8WX,1.0,null,0.0
S0011,SKU00017,2025-06-30,4.0,29.09,0.0
S0021,SKU00099,2025-06-30,2.0,93.89,0.0
S0015,SKU00100,2025-06-30,3.0,50.33,0.0


In [ ]:
%sql
-- Row counts
SELECT COUNT(*) FROM ishi_catalog.bronze.pos;

count(1)
5229


In [ ]:
from pyspark.sql.functions import col
import re

def clean_column_names(df):
    for c in df.columns:
        new_c = re.sub(r'[^a-zA-Z0-9_]', '_', c)
        df = df.withColumnRenamed(c, new_c)
    return df

stores_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/stores"
stores_schema = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_schemas/stores"
stores_chkpt  = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_checkpoints/stores"
bronze_stores = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/stores"

stores_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", stores_schema)
    .load(stores_source)
)

stores_df = clean_column_names(stores_df)

(stores_df.writeStream
    .format("delta")
    .option("checkpointLocation", stores_chkpt)
    .option("path", bronze_stores)
    .trigger(once=True)
    .outputMode("append")
    .start())


In [ ]:
inventory_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/inventory"
inventory_schema = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_schemas/inventory"
inventory_chkpt  = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_checkpoints/inventory"
bronze_inventory = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/inventory"

inventory_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", inventory_schema)
    .load(inventory_source)
)

inventory_df = clean_column_names(inventory_df)

(inventory_df.writeStream
    .format("delta")
    .option("checkpointLocation", inventory_chkpt)
    .option("path", bronze_inventory)
    .trigger(once=True)
    .outputMode("append")
    .start())


In [ ]:
holidays_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/holidays"
holidays_schema = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_schemas/holidays"
holidays_chkpt  = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_checkpoints/holidays"
bronze_holidays = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/holidays"

holidays_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", holidays_schema)
    .load(holidays_source)
)

holidays_df = clean_column_names(holidays_df)

(holidays_df.writeStream
    .format("delta")
    .option("checkpointLocation", holidays_chkpt)
    .option("path", bronze_holidays)
    .trigger(once=True)
    .outputMode("append")
    .start())


In [ ]:
weather_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/weather"
weather_schema = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_schemas/weather"
weather_chkpt  = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_checkpoints/weather"
bronze_weather = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/weather"

weather_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", weather_schema)
    .load(weather_source)
)

weather_df = clean_column_names(weather_df)

(weather_df.writeStream
    .format("delta")
    .option("checkpointLocation", weather_chkpt)
    .option("path", bronze_weather)
    .trigger(once=True)
    .outputMode("append")
    .start())


In [ ]:
products_source = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/landing/products"
products_schema = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_schemas/products"
products_chkpt  = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/_checkpoints/products"
bronze_products = "abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/products"

products_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", products_schema)
    .load(products_source)
)

products_df = clean_column_names(products_df)

(products_df.writeStream
    .format("delta")
    .option("checkpointLocation", products_chkpt)
    .option("path", bronze_products)
    .trigger(once=True)
    .outputMode("append")
    .start())


### **Registering the Delta paths as managed tables in Unity Catalog (ishi_catalog)**

In [ ]:
%sql
-- Example: list all catalogs
SHOW CATALOGS;



catalog
atg_catalog
basab_catalog_retail
hive_metastore
ish_catalog
ishi_catalog
ishika_29aug_catalog
ishika_catalog
ishika_catalog_2030
ishika_new_catalog
keshcat


In [ ]:
%sql
-- Example: switch to your catalog
USE CATALOG ishi_catalog;

In [ ]:
%sql
-- Example: list all schemas
SHOW SCHEMAS;


databaseName
bronze
default
gold
information_schema
landing
silver


## Mapping each delta path to SQL tables in unity catalog

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ishi_catalog.bronze.pos
USING DELTA
LOCATION 'abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/pos';


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ishi_catalog.bronze.stores
USING DELTA
LOCATION 'abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/stores';

CREATE TABLE IF NOT EXISTS ishi_catalog.bronze.inventory
USING DELTA
LOCATION 'abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/inventory';

CREATE TABLE IF NOT EXISTS ishi_catalog.bronze.holidays
USING DELTA
LOCATION 'abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/holidays';

CREATE TABLE IF NOT EXISTS ishi_catalog.bronze.weather
USING DELTA
LOCATION 'abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/weather';

CREATE TABLE IF NOT EXISTS ishi_catalog.bronze.products
USING DELTA
LOCATION 'abfss://capstonecontainer@ishikastorage.dfs.core.windows.net/bronze/products';


In [ ]:
%sql
-- Check the tables
SHOW TABLES IN ishi_catalog.bronze;


database,tableName,isTemporary
bronze,holidays,false
bronze,inventory,false
bronze,pos,false
bronze,products,false
bronze,stores,false
bronze,weather,false
,_sqldf,true


In [ ]:
%sql

-- Sample data
SELECT * FROM ishi_catalog.bronze.pos LIMIT 10;


store_id,sku_id,date,units_sold,price,promo_flag,_rescued_data
S0014,SKU00146,2025-06-30,5.0,1.13,0.0,null
S0050,SKU00052,2025-06-30,3.0,29.55,0.0,null
S0036,SKU00035,2025-06-30,150.37390943082676,1.43,1.0,null
S0015,SKU00162,2025-06-30,4.0,10.47,0.0,null
S0042,SKU00166,2025-06-30,3.0,null,1.0,null
S0042,SKU00071,2025-06-30,6.0,2517.1220398836726,1.0,null
S0037,SKU00059,INVALID_8WX,1.0,null,0.0,null
S0011,SKU00017,2025-06-30,4.0,29.09,0.0,null
S0021,SKU00099,2025-06-30,2.0,93.89,0.0,null
S0015,SKU00100,2025-06-30,3.0,50.33,0.0,null


In [ ]:
%sql
-- Row counts
SELECT COUNT(*) FROM ishi_catalog.bronze.stores;


count(1)
52


In [ ]:
%sql
-- Sample data
SELECT * FROM ishi_catalog.bronze.pos LIMIT 50;

store_id,sku_id,date,units_sold,price,promo_flag,_rescued_data
S0014,SKU00146,2025-06-30,5.0,1.13,0.0,null
S0050,SKU00052,2025-06-30,3.0,29.55,0.0,null
S0036,SKU00035,2025-06-30,150.37390943082676,1.43,1.0,null
S0015,SKU00162,2025-06-30,4.0,10.47,0.0,null
S0042,SKU00166,2025-06-30,3.0,null,1.0,null
S0042,SKU00071,2025-06-30,6.0,2517.1220398836726,1.0,null
S0037,SKU00059,INVALID_8WX,1.0,null,0.0,null
S0011,SKU00017,2025-06-30,4.0,29.09,0.0,null
S0021,SKU00099,2025-06-30,2.0,93.89,0.0,null
S0015,SKU00100,2025-06-30,3.0,50.33,0.0,null


In [ ]:
from pyspark.sql.functions import col, trim, upper, to_date, regexp_replace
from pyspark.sql.types import IntegerType, DoubleType

# Set catalog and schema
spark.sql("USE CATALOG ishi_catalog")
spark.sql("USE SCHEMA bronze")


DataFrame[]

In [ ]:
# 1. Read Bronze POS table
pos_bronze_df = spark.table("ishi_catalog.bronze.pos")

In [ ]:
%sql
SHOW TABLES IN ishi_catalog.bronze;


database,tableName,isTemporary
bronze,holidays,false
bronze,inventory,false
bronze,pos,false
bronze,products,false
bronze,stores,false
bronze,weather,false


### pos table tranformations ( Silver Layer)

In [ ]:
from pyspark.sql.functions import col, when, trim
from pyspark.sql.types import IntegerType, DoubleType, DateType
import pyspark.sql.functions as F

# Read Bronze POS table
pos_df = spark.table("ishi_catalog.bronze.pos")

# Clean POS data
pos_silver_df = (
    pos_df
    # Trim string fields
    .withColumn("store_id", trim(col("store_id")))
    .withColumn("sku_id", trim(col("sku_id")))

    # Remove invalid IDs
    .filter(~col("store_id").rlike("INVALID|NULL") & col("store_id").isNotNull())
    .filter(~col("sku_id").rlike("INVALID|NULL") & col("sku_id").isNotNull())

    # Date cleansing
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
    .filter(col("date").isNotNull())  # drop invalid dates

    # Units sold cleanup (convert to int, drop negatives/nulls)
    .withColumn("units_sold", col("units_sold").cast(IntegerType()))
    .filter(col("units_sold").isNotNull() & (col("units_sold") >= 0))

    # Price cleanup (convert to double, drop nulls/negatives/outliers)
    .withColumn("price", col("price").cast(DoubleType()))
    .filter(col("price").isNotNull() & (col("price") > 0) & (col("price") < 1000))

    # Promo flag cleanup (cast to int, keep only 0/1, else set to 0)
    .withColumn("promo_flag",
                when(col("promo_flag").isin("1", "0"), col("promo_flag").cast(IntegerType()))
                .otherwise(F.lit(0)))

    .drop("_rescued_data")
)

# Write Silver POS table
(
    pos_silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ishi_catalog.silver.pos")
)

print("Silver POS table created successfully!")


Silver POS table created successfully!


In [ ]:
%sql
select * from ishi_catalog.silver.pos LIMIT 50;

store_id,sku_id,date,units_sold,price,promo_flag
S0014,SKU00146,2025-06-30,5,1.13,0
S0050,SKU00052,2025-06-30,3,29.55,0
S0036,SKU00035,2025-06-30,150,1.43,0
S0015,SKU00162,2025-06-30,4,10.47,0
S0011,SKU00017,2025-06-30,4,29.09,0
S0021,SKU00099,2025-06-30,2,93.89,0
S0015,SKU00100,2025-06-30,3,50.33,0
S0029,SKU00153,2025-06-30,2,93.93,0
S0016,SKU00188,2025-06-30,3,49.54,0
S0038,SKU00009,2025-06-30,4,63.17,0


In [ ]:
%sql
-- Row counts
SELECT COUNT(*) FROM ishi_catalog.silver.pos;

count(1)
4007


In [ ]:
%sql
select* from ishi_catalog.bronze.weather limit 50;

date,region,temperature_c,rainfall_mm,_rescued_data
INVALID_GH4,North,26.7,28.1,null
2025-06-30,INVALID_9A3,27.1,49.5,null
2025-06-30,East,22.9,15.4,null
2025-06-30,West,34.5,31.7,null
2025-07-01,null,18.4,44.9,null
2025-07-01,South,10.3,23.4,null
2025-07-01,East,18.3,40.7,null
null,West,29.5,24.8,null
2025-07-02,North,27.4,17.9,null
2025-07-02,South,32.7,31.5,null


In [ ]:
%sql
-- Row counts
SELECT COUNT(*) FROM ishi_catalog.bronze.weather;

count(1)
252


### Weather table transformations

In [ ]:
from pyspark.sql.functions import col, to_date, when
from pyspark.sql.types import DoubleType

# Read Bronze Weather
weather_bronze = spark.table("ishi_catalog.bronze.weather")

# Cleanse the data
weather_silver = (
    weather_bronze
    # 1. Convert date, drop invalid/null
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .filter(col("date").isNotNull())

    # 2. Clean region (drop rows where region is null/invalid)
    .filter(col("region").isNotNull())
    .filter(~col("region").like("INVALID%"))

    # 3. Cast numeric columns
    .withColumn("temperature_c", col("temperature_c").cast(DoubleType()))
    .withColumn("rainfall_mm", col("rainfall_mm").cast(DoubleType()))

    # 4. Filter out unrealistic values
    .filter((col("temperature_c") > -50) & (col("temperature_c") < 60))
    .filter((col("rainfall_mm") >= 0) & (col("rainfall_mm") < 500))  # rainfall capped

    # 5. Drop rescued_data
    .drop("_rescued_data")
)

# Write Silver Weather table
(
    weather_silver.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "false")
    .saveAsTable("ishi_catalog.silver.weather")
)

print(" Silver Weather table created successfully!")


✅ Silver Weather table created successfully!


In [ ]:
%sql
select* from ishi_catalog.silver.weather limit 50;

date,region,temperature_c,rainfall_mm
2025-06-30,East,22.9,15.4
2025-06-30,West,34.5,31.7
2025-07-01,South,10.3,23.4
2025-07-01,East,18.3,40.7
2025-07-02,North,27.4,17.9
2025-07-02,South,32.7,31.5
2025-07-02,East,11.8,1.9
2025-07-03,North,12.4,10.7
2025-07-03,South,12.7,1.9
2025-07-03,West,17.4,47.2


In [ ]:
%sql
-- Row counts
SELECT COUNT(*) FROM ishi_catalog.silver.weather;

count(1)
198


## Holiday table transformation

In [ ]:
# 1. Read Bronze holidays table
pos_bronze_df = spark.table("ishi_catalog.bronze.holidays")

In [ ]:
%sql
select* from ishi_catalog.bronze.holidays limit 50

date,event_name,_rescued_data
2025-08-18,Easter,null
2025-09-24,Diwali,null
2025-08-28,LaborDay,null
2025-08-07,Christmas,null
2025-09-11,Easter,null
2025-08-27,LaborDay,null
2025-09-03,Christmas,null
2025-09-16,Easter,null
2025-08-04,Diwali,null
2025-08-21,Christmas,null


In [ ]:
from pyspark.sql.functions import lit, upper, trim, to_date

# Create a correct holiday mapping
holiday_mapping = [
    ("2025-12-25", "CHRISTMAS"),
    ("2025-11-01", "DIWALI"),    # adjust according to actual 2025 Diwali date
    ("2025-05-05", "EASTER"),    # adjust according to actual 2025 Easter date
    ("2025-09-01", "LABORDAY")   # adjust according to actual 2025 Labor Day
]

# Convert to Spark DataFrame
holidays_correct = spark.createDataFrame(holiday_mapping, ["date", "event_name"])

# Convert date to proper type
holidays_silver = holidays_correct.withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
                                   .withColumn("event_name", upper(trim(col("event_name"))))

# Save to Silver
(
    holidays_silver.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "false")
    .saveAsTable("ishi_catalog.silver.holidays")
)

print(" Silver Holidays table created with correct dates!")



 Silver Holidays table created with correct dates!


In [ ]:
%sql
select* from ishi_catalog.silver.holidays limit 50

date,event_name
2025-12-25,CHRISTMAS
2025-11-01,DIWALI
2025-05-05,EASTER
2025-09-01,LABORDAY


## Stores table transformation

In [ ]:
# 1. Read Bronze stores table
pos_bronze_df = spark.table("ishi_catalog.bronze.stores")

In [ ]:
%sql
select* from ishi_catalog.bronze.stores limit 50

store_id,region,format,size_sqft,opening_date,_rescued_data
S0001,East,Hypermarket,7468.0,2017-04-20,null
S0002,North,null,9779.0,2011-01-21,null
S0003,East,Convenience,1950.0,2015-09-10,null
S0004,South,Hypermarket,278441.8367346939,2014-11-12,null
S0005,West,Hypermarket,4943.0,2011-01-07,null
S0006,West,Hypermarket,3028.0,2012-07-03,null
S0007,North,Convenience,7499.0,2010-07-23,null
S0008,South,Hypermarket,3181.0,2013-04-01,null
S0009,West,Hypermarket,9858.0,2011-04-28,null
S0010,East,Convenience,3961.0,2011-02-27,null


In [ ]:
from pyspark.sql.functions import col, trim, upper, row_number
from pyspark.sql.window import Window

# Load Bronze stores table
stores_bronze = spark.table("ishi_catalog.bronze.stores")

# Clean stores data
stores_silver = (
    stores_bronze
    # Remove invalid store_id, region, and format
    .filter((col("store_id").isNotNull()) & (~col("store_id").startswith("INVALID")))
    .filter((col("region").isNotNull()) & (~col("region").startswith("INVALID")))
    .filter((col("format").isNotNull()) & (~col("format").startswith("INVALID")))

    # Standardize region and format
    .withColumn("region", upper(trim(col("region"))))
    .withColumn("format", upper(trim(col("format"))))

    # Convert opening_date to proper date format
    .withColumn("opening_date", to_date(col("opening_date"), "yyyy-MM-dd"))
    .filter(col("opening_date").isNotNull())

    # Filter size_sqft
    .withColumn("size_sqft", when((col("size_sqft") > 0) & (col("size_sqft") < 50000), col("size_sqft")).otherwise(None))
    .filter(col("size_sqft").isNotNull())

    # Remove duplicates keeping first occurrence
    .withColumn("rn", row_number().over(Window.partitionBy("store_id").orderBy("opening_date")))
    .filter(col("rn") == 1)
    .drop("rn", "_rescued_data")
)

# Write to Silver
(
    stores_silver.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("ishi_catalog.silver.stores")
)

print(" Silver Stores table updated with clean format and region!")



 Silver Stores table updated with clean format and region!


In [ ]:
%sql
select* from ishi_catalog.silver.stores limit 200

store_id,region,format,size_sqft,opening_date
S0001,EAST,HYPERMARKET,7468.0,2017-04-20
S0003,EAST,CONVENIENCE,1950.0,2015-09-10
S0005,WEST,HYPERMARKET,4943.0,2011-01-07
S0006,WEST,HYPERMARKET,3028.0,2012-07-03
S0007,NORTH,CONVENIENCE,7499.0,2010-07-23
S0008,SOUTH,HYPERMARKET,3181.0,2013-04-01
S0009,WEST,HYPERMARKET,9858.0,2011-04-28
S0010,EAST,CONVENIENCE,3961.0,2011-02-27
S0011,SOUTH,SUPERMARKET,2596.0,2016-02-22
S0013,SOUTH,SUPERMARKET,9711.0,2014-10-18


## Inventory table transformation

In [ ]:
# 1. Read Bronze inventory table
pos_bronze_df = spark.table("ishi_catalog.bronze.inventory")

In [ ]:
%sql
select* from ishi_catalog.bronze.inventory limit 50

store_id,sku_id,stock_level,_rescued_data
S0048,SKU00145,18.0,null
S0046,SKU00001,335.0,null
S0038,SKU00123,179.0,null
null,null,435.0,null
S0006,SKU00090,304.0,null
S0021,SKU00125,85.0,null
S0021,SKU00171,267.0,null
S0042,INVALID_0BZ,100.0,null
S0032,SKU00131,55.0,null
S0049,SKU00168,256.0,null


In [ ]:
from pyspark.sql.functions import col, when

# Load Bronze inventory table
inventory_bronze = spark.table("ishi_catalog.bronze.inventory")

# Clean inventory data
inventory_silver = (
    inventory_bronze
    # Remove null or invalid store_id/sku_id
    .filter((col("store_id").isNotNull()) & (~col("store_id").startswith("INVALID")))
    .filter((col("sku_id").isNotNull()) & (~col("sku_id").startswith("INVALID")))

    # Remove null stock_level
    .filter(col("stock_level").isNotNull())

    # Remove outliers in stock_level (example threshold: >500)
    .filter((col("stock_level") >= 0) & (col("stock_level") <= 500))

    # Drop _rescued_data
    .drop("_rescued_data")
)

# Write cleaned Silver inventory table
(
    inventory_silver.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("ishi_catalog.silver.inventory")
)

print(" Silver Inventory table cleaned and saved!")


 Silver Inventory table cleaned and saved!


In [ ]:
%sql
select* from ishi_catalog.silver.inventory limit 50

store_id,sku_id,stock_level
S0048,SKU00145,18.0
S0046,SKU00001,335.0
S0038,SKU00123,179.0
S0006,SKU00090,304.0
S0021,SKU00125,85.0
S0021,SKU00171,267.0
S0032,SKU00131,55.0
S0049,SKU00168,256.0
S0032,SKU00158,104.0
S0020,SKU00072,163.0


## Products table transformation

In [ ]:
# 1. Read Bronze products table
pos_bronze_df = spark.table("ishi_catalog.bronze.products")

In [ ]:
%sql
select* from ishi_catalog.bronze.products limit 50

sku_id,category,subcategory,brand,_rescued_data
SKU00001,PersonalCare,Shampoo,Brand_3FA8,null
SKU00002,Produce,Vegetable,Brand_FTAT,null
SKU00003,Beverages,Water,Brand_4S6M,null
SKU00004,Snacks,Nuts,Brand_TSWW,null
null,Snacks,Chips,INVALID_HLM,null
SKU00006,PersonalCare,INVALID_OZQ,Brand_BBKJ,null
SKU00007,Produce,Vegetable,Brand_Q798,null
SKU00008,PersonalCare,Soap,Brand_HIHH,null
SKU00009,Produce,Grains,Brand_4RX2,null
null,Beverages,Water,Brand_ENZH,null


In [ ]:
from pyspark.sql.functions import col, when

# Assuming your raw DataFrame is called df_products_raw
products_silver = products_bronze.filter(
    (~col('sku_id').startswith('INVALID')) & col('sku_id').isNotNull()
)

# Fill null values with "Unknown"
products_silver = products_silver.fillna({
    "category": "Unknown",
    "subcategory": "Unknown",
    "brand": "Unknown"
})

# Replace invalid entries in category, subcategory, brand
products_silver = products_silver.withColumn(
    "category", when(col("category").startswith("INVALID"), "Unknown").otherwise(col("category"))
).withColumn(
    "subcategory", when(col("subcategory").startswith("INVALID"), "Unknown").otherwise(col("subcategory"))
).withColumn(
    "brand", when(col("brand").startswith("INVALID"), "Unknown").otherwise(col("brand"))
)

# Drop the _rescued_data column
products_silver = products_silver.drop("_rescued_data")

# Display the cleaned DataFrame
products_silver.display()


sku_id,category,subcategory,brand
SKU00001,PersonalCare,Shampoo,Brand_3FA8
SKU00002,Produce,Vegetable,Brand_FTAT
SKU00003,Beverages,Water,Brand_4S6M
SKU00004,Snacks,Nuts,Brand_TSWW
SKU00006,PersonalCare,Unknown,Brand_BBKJ
SKU00007,Produce,Vegetable,Brand_Q798
SKU00008,PersonalCare,Soap,Brand_HIHH
SKU00009,Produce,Grains,Brand_4RX2
SKU00012,Household,Cleaner,Brand_O80G
SKU00013,Snacks,Chips,Brand_F63F
